In [1]:
!pip install gradio

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import re
import gradio as gr
import torch
import pandas as pd

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [5]:
from sklearn.preprocessing import LabelEncoder

df_train = pd.read_csv('/content/train_cleaned_checkpoint.csv')

label_encoder = LabelEncoder()
label_encoder.fit(df_train['label'])

# Class Mapping
class_mapping = dict(zip(label_encoder.classes_, range(len(label_encoder.classes_))))
print("Emotion Class Mapping:", class_mapping)

Emotion Class Mapping: {'anger': 0, 'fear': 1, 'happy': 2, 'love': 3, 'sadness': 4}


In [6]:
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer
from safetensors.torch import load_file

class IndoBERTweetCustom(nn.Module):
    def __init__(self, num_labels):
        super(IndoBERTweetCustom, self).__init__()
        self.num_labels = num_labels
        self.bert = AutoModel.from_pretrained("indolem/indobertweet-base-uncased")

        # Arsitektur Klasifikasi Custom (Shaw et al.)
        self.dense1 = nn.Linear(self.bert.config.hidden_size, 64)
        self.dropout = nn.Dropout(0.05)
        self.dense2 = nn.Linear(64, 16)
        self.classifier = nn.Linear(16, num_labels)

    def forward(self, input_ids, attention_mask, labels=None, **kwargs):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)

        # GLOBAL AVERAGE POOLING
        mask_expanded = attention_mask.unsqueeze(-1).expand(outputs.last_hidden_state.size()).float()
        sum_embeddings = torch.sum(outputs.last_hidden_state * mask_expanded, 1)
        sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)
        pooled_output = sum_embeddings / sum_mask

        # CUSTOM CLASSIFICATION HEAD
        x = self.dense1(pooled_output)
        x = torch.relu(x)
        x = self.dropout(x)
        x = self.dense2(x)
        x = torch.relu(x)
        logits = self.classifier(x)

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))
            return {"loss": loss, "logits": logits}

        return {"logits": logits}

model_tweet = IndoBERTweetCustom(
    num_labels=len(class_mapping),
).to(device)

model_path = "/content/drive/MyDrive/indobertweet-emotion-v1/model.safetensors"
weight = load_file(model_path)
model_tweet.load_state_dict(weight)
model_tweet.to(device)

folder_path = "/content/drive/MyDrive/indobertweet-emotion-v1/"
tokenizer_tw = AutoTokenizer.from_pretrained(folder_path)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.10k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: indolem/indobertweet-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

In [7]:
def prep_indobertweet(text):
    text = str(text).lower()
    text = re.sub(r'\[username\]', '@USER', text)
    text = re.sub(r'\[url\]', 'HTTPURL', text)
    return text.strip()

In [8]:
import gradio as gr
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F

model_tweet.eval()

# 1. Single Prediction
def single_prediction(text):
    if not text.strip():
        return {"No Input": 1.0}

    cleaned_text = prep_indobertweet(text)
    inputs = tokenizer_tw(cleaned_text, return_tensors="pt", truncation=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model_tweet(**inputs)
        logits = outputs["logits"]

    prob = F.softmax(logits, dim=1).squeeze().cpu().numpy()
    class_name = label_encoder.classes_
    predictions = {class_name[i]: float(prob[i]) for i in range(len(class_name))}

    return predictions

# 2. Batch Prediction
def batch_prediction(file_csv, column_name):
    # 1. Input Validation
    if file_csv is None:
        return None, "[ERROR]: Upload CSV file first!"

    try:
        df = pd.read_csv(file_csv.name)
    except Exception as e:
        return None, f"[ERROR]: Failed to read the CSV. ({e})"

    if column_name not in df.columns:
        return None, f"[ERROR]: '{column_name}' column not found! Available column: {', '.join(df.columns)}"

    # 2. Ekstraksi Data
    list_text = df[column_name].dropna().astype(str).tolist()
    if len(list_text) == 0:
        return None, "[ERROR]: Empty Column."

    # 3. Batching
    batch_size = 32
    full_prediction = []

    for i in range(0, len(list_text), batch_size):
        batch_text = list_text[i : i + batch_size]
        # Pembersihan text
        cleaned_batch = [prep_indobertweet(text) for text in batch_text]

        # Tokenisasi batch (wajib pakai padding di sini agar panjang matriks seragam)
        inputs = tokenizer_tw(
            cleaned_batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=43
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        # Prediksi
        with torch.no_grad():
            outputs = model_tweet(**inputs)
            # Ambil indeks kelas probabilitas tertinggi
            kelas_tertinggi = torch.argmax(outputs["logits"], dim=1).cpu().numpy()
            full_prediction.extend(kelas_tertinggi)

    # 4. Konversi indeks ke label text
    class_name = label_encoder.classes_
    prediction_label = [class_name[p] for p in full_prediction]

    # 5. Visualisasi Pie Chart
    distribution = pd.Series(prediction_label).value_counts()

    fig, ax = plt.subplots(figsize=(7, 7))
    warna = sns.color_palette("pastel")[0:len(distribution)]

    ax.pie(
        distribution,
        labels=distribution.index,
        autopct='%1.1f%%',
        startangle=140,
        colors=warna,
        wedgeprops={'edgecolor': 'white', 'linewidth': 2}
    )
    ax.set_title(f'Emotion Distribution from {len(list_text)} Data', fontsize=14, pad=20, fontweight='bold')

    return fig, "[SUCCESS]"

# App
with gr.Blocks(theme=gr.themes.Soft()) as app:
    gr.Markdown("# Indonesian Text Emotion Classification (IndoBERTweet)")
    gr.Markdown("## This system will predict text based on 5 emotion classes: anger, fear, happiness, love, and sadness.")

    # SINGLE PREDICTION
    with gr.Tab("Single Text Prediction"):
        with gr.Row():
            with gr.Column(scale=2):
                input_text = gr.Textbox(lines=4, placeholder="Input text here...", label="Input Text")
                single_btn = gr.Button("Analyze Emotion", variant="primary")
            with gr.Column(scale=1):
                label_out = gr.Label(num_top_classes=5, label="Prediction Confidence")

        single_btn.click(fn=single_prediction, inputs=input_text, outputs=label_out)

    # BATCH Evaluation
    with gr.Tab("Batch Evaluation (CSV)"):
        gr.Markdown("Upload `.csv` file, specify the name of the column containing the text, and the system will process the entire row automatically.")

        with gr.Row():
            with gr.Column(scale=1):
                file_in = gr.File(label="Upload File (.csv)", file_types=[".csv"])
                column_in = gr.Textbox(value="text", label="Column containing text", placeholder="eg.: tweet, comment, teks, etc")
                batch_btn = gr.Button("Start Batch Evaluation", variant="primary")
                status_out = gr.Textbox(label="Stats", interactive=False, lines=3)

            with gr.Column(scale=2):
                plot_out = gr.Plot(label="Results")

        batch_btn.click(
            fn=batch_prediction,
            inputs=[file_in, column_in],
            outputs=[plot_out, status_out]
        )

app.launch(share=True)

/tmp/ipykernel_811/4055458600.py:97: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://010e6537a0034f8c48.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
